## Executive Summary

This notebook computes the **magnetomotive force (MMF) waveform and harmonic spectrum** produced by an AC stator winding.

### Why It Matters
The spatial shape of the airgap MMF directly determines:
- **Torque production** — the working harmonic (ν = p) drives the fundamental torque.
- **Torque ripple** — sub- and super-harmonics create pulsating torque components.
- **Rotor losses** — higher-order harmonics induce eddy currents in the rotor magnets and back-iron, causing heating and efficiency loss.
- **Acoustic noise** — MMF harmonics that coincide with structural resonances generate audible noise.

Designers use the MMF spectrum to compare winding configurations, choose slot/pole combinations, and decide between single- and double-layer layouts before running finite-element analysis.

## How It Fits Into Motor Design

```
emachines workflow:

  01_winding.ipynb
  (winding factors, star-of-slots)
        │
        ▼
  01_winding_mmf.ipynb   ←── YOU ARE HERE
  (MMF waveform & spectrum)
        │
        ▼
  02_pmsm.ipynb          (torque, back-EMF via kw and MMF)
  04_mec_solver.ipynb    (flux linkage in magnetic circuit)
```

**Depends on:** `01_winding.ipynb` — uses `build_coil_matrix`, `get_valid_coil_spans` from `emachines.winding.sos`.

**Feeds into:** `02_pmsm.ipynb`, `04_mec_solver.ipynb`.

## What You Provide (Inputs)

| Parameter | Symbol | Description | Units | Typical Range |
|-----------|--------|-------------|-------|---------------|
| Stator slots | Q | Total number of stator slots | — | 6 – 72 |
| Poles | P | Number of poles (even) | — | 2 – 20 |
| Phases | m | Number of phases | — | 3 (three-phase) |
| Layers | layers | Single- or double-layer winding | — | 1 or 2 |
| Coil span | w | Span of each coil in slots | slots | 1 – Q/p |
| Time angle | θ_e | Electrical time angle | rad | 0 – 2π |
| Mode | mode | `"three_phase"`, `"single_phase"`, `"winding_factor"` | — | — |
| Max harmonic | nu_max | Highest spatial harmonic to evaluate | — | 20 – 50 |

## What You Get (Outputs)

| Result | Symbol | Meaning |
|--------|--------|---------|
| Conductor distribution | n[i] | Signed A·turns per slot at angle θ_e |
| MMF waveform | F(θ) | Staircase MMF along the airgap |
| Winding factor spectrum | kw(ν) | |kw| for each mechanical harmonic |
| MMF harmonic amplitudes | F̂(ν) | kw(ν)·p/ν, balanced 3-phase cancellation applied |

### Quick Example
```python
from emachines.winding.mmf import mmf_waveform, mmf_harmonics

# 12-slot / 10-pole double-layer FSCW at θ_e = 0
angles, F = mmf_waveform(12, 10, layers=2)

# Three-phase MMF harmonic spectrum, normalised
nu, amp = mmf_harmonics(12, 10, layers=2)
```

## Abbreviations and Notation

| Symbol | Definition | Units |
|--------|-----------|-------|
| Q | Number of stator slots | — |
| P | Number of poles | — |
| p | Pole pairs, p = P/2 | — |
| m | Number of phases | — |
| ν | Mechanical spatial harmonic order | — |
| w | Coil span in slots | slots |
| θ_e | Electrical time angle | rad |
| n_i | Signed ampere-conductor value at slot i | A·turns (pu) |
| F(θ) | MMF waveform along the airgap | A·turns (pu) |
| kw(ν) | Winding factor magnitude at harmonic ν | — |
| F̂(ν) | MMF harmonic amplitude = kw(ν)·p/ν | A·turns (pu) |
| t | Winding periodicity = gcd(Q, p) | — |

## Imports and Setup

In [ ]:
#| default_exp winding.mmf
#| export
from __future__ import annotations

import numpy as np

from emachines.winding.sos import build_coil_matrix
from emachines.winding.winding_factors import _optimal_coil_span, winding_factor_spectrum

## Background: MMF in AC Windings

When alternating current flows through a stator winding, each conductor contributes a localised magnetomotive force proportional to the current it carries and its winding direction. Summing these contributions around the airgap circumference via Ampere's law yields the **MMF waveform** F(θ).

For a sinusoidal balanced three-phase supply:
$$
i_k(\theta_e) = \hat{I}\cos\!\left(\theta_e - \frac{2\pi k}{m}\right),\quad k = 0,1,\ldots,m-1
$$

The MMF waveform is computed in three steps:

1. **Slot assignment** — `build_coil_matrix` maps each slot conductor to a phase and current direction.
2. **Ampere-conductor distribution** — multiply signed conductor occupancy by instantaneous phase currents to get n[i] at each slot.
3. **Cumulative sum** — integrate n[i] around the circumference (Ampere's law) and remove the DC offset.

The resulting staircase is then decomposed by spatial Fourier analysis into its harmonic spectrum.

---

## `mmf_distribution`: Ampere-Conductor Distribution

### Theory

The **ampere-conductor distribution** n(θ) is a step function that equals the net signed conductor current at each slot position:

$$
n_i = \sum_{\text{layer}} \operatorname{sign}(\text{code}_{i,\text{layer}}) \cdot i_{\,|\text{code}|-1}(\theta_e)
$$

where `code` is the signed conductor occupancy from `build_coil_matrix` (+k = phase k forward, −k = phase k return), and $i_k(\theta_e)$ is the instantaneous current of phase k.

**Physical meaning:** n(θ) is the "source density" of MMF. The MMF itself is its spatial running integral (cumulative sum around the bore).

**Design insight:** The distribution of conductors across slots determines which spatial harmonics are present and how strongly. Distributed windings spread conductors to suppress low-order harmonics; concentrated windings concentrate them for high torque density at the cost of richer harmonic content.

**References:** Pyrhönen et al. (2014), §2.4; Müller, Vogt & Ponick (2008), §3.4.

### `mmf_distribution`: Implementation

In [ ]:
#| export
def _phase_currents(m: int, theta_electrical: float) -> np.ndarray:
    """
    Instantaneous balanced m-phase currents (per-unit peak = 1).

    .. math::
        i_k = \\cos(\\theta_e - 2\\pi k / m), \\quad k = 0, 1, \\ldots, m-1

    Parameters
    ----------
    m                : int    Number of phases
    theta_electrical : float  Electrical time angle (radians)

    Returns
    -------
    np.ndarray, shape (m,)
    """
    k = np.arange(m, dtype=np.float64)
    return np.cos(theta_electrical - 2.0 * np.pi * k / m)

In [ ]:
#| export
def mmf_distribution(
    Q: int,
    P: int,
    m: int = 3,
    layers: int = 1,
    w: int | None = None,
    theta_electrical: float = 0.0,
) -> np.ndarray:
    r"""
    Instantaneous signed ampere-conductor distribution along the airgap.

    For each slot position *i* (mechanical angle θᵢ = 2π·i/Q), ``n[i]``
    is the sum of signed conductor currents across all winding layers:

    .. math::
        n_i = \sum_{\text{layer}} \operatorname{sign}(\text{code}_{i,\text{layer}})
              \cdot i_{\,|\text{code}|-1}(\theta_e)

    Parameters
    ----------
    Q                : int    Number of stator slots
    P                : int    Number of poles (even, ≥ 2)
    m                : int    Number of phases (default 3)
    layers           : int    1 (single-layer) or 2 (double-layer)
    w                : int | None
                              Coil span in slots. ``None`` → coil span that
                              maximises the fundamental winding factor.
    theta_electrical : float  Electrical time angle (radians, default 0).

    Returns
    -------
    np.ndarray, shape (Q,), dtype float64
        Signed ampere-conductor value at each slot. Per-unit (peak = 1 A·turn).

    Examples
    --------
    >>> n = mmf_distribution(12, 10, layers=2)
    >>> n.shape
    (12,)
    >>> abs(n.sum()) < 1e-10
    True
    """
    if w is None:
        w = _optimal_coil_span(Q, P, m, layers)

    matrix = build_coil_matrix(Q, P, m, layers, w)   # shape (layers, Q)
    i_ph = _phase_currents(m, theta_electrical)        # shape (m,)

    n = np.zeros(Q, dtype=np.float64)
    for lyr in range(layers):
        for i in range(Q):
            code = int(matrix[lyr, i])
            if code == 0:
                continue
            k = abs(code) - 1          # 0-based phase index
            sign = 1 if code > 0 else -1
            n[i] += sign * i_ph[k]

    return n

### `mmf_distribution`: Example

In [ ]:
from emachines.winding.mmf import mmf_distribution
import numpy as np

# 12s/10p double-layer FSCW at θ_e = 0  (phase A at positive peak)
n = mmf_distribution(12, 10, layers=2, theta_electrical=0.0)

print("12-slot / 10-pole double-layer FSCW, θ_e = 0°")
print(f"Slot conductors: {np.round(n, 4)}")
print(f"Sum (should be ≈ 0): {n.sum():.2e}")
print()
print("Interpretation: positive = net A·turns in forward direction,")
print("                negative = net A·turns in return direction.")

---

## `mmf_waveform`: MMF Staircase

### Theory

The MMF waveform is obtained by applying Ampere's law around the stator bore. Integrating the ampere-conductor distribution n(θ) in the positive θ direction gives:

$$
F(\theta_i) = \sum_{j=0}^{i-1} n_j - \overline{F}
$$

where $\overline{F}$ is the spatial mean, subtracted to enforce zero net MMF around the closed path (no DC offset).

The result is a **staircase waveform** — constant between slots, stepping at each slot opening. In the limit of many slots (large Q) it approaches a sinusoid; for concentrated windings with few slots per phase it is highly non-sinusoidal.

**Design insight:** The ratio of fundamental amplitude to harmonic content directly determines torque quality. A near-sinusoidal MMF (many slots, distributed winding) gives low ripple; a square-wave MMF (concentrated winding) is rich in harmonics but simpler to manufacture.

**References:** Pyrhönen et al. (2014), §2.4, eq. (2.73); Müller, Vogt & Ponick (2008), §3.4.

### `mmf_waveform`: Implementation

In [ ]:
#| export
def mmf_waveform(
    Q: int,
    P: int,
    m: int = 3,
    layers: int = 1,
    w: int | None = None,
    theta_electrical: float = 0.0,
) -> tuple[np.ndarray, np.ndarray]:
    r"""
    Instantaneous MMF staircase waveform along the airgap.

    Applies Ampere's law around the stator bore: the MMF at mechanical
    angle θ equals the cumulative sum of ampere-conductors from slot 0
    up to the slot preceding θ, with the DC offset removed.

    .. math::
        F(\theta_i) = \sum_{j=0}^{i-1} n_j - \overline{F}

    The staircase is returned *closed* (first point repeated at 360°)
    for direct use in step-line plots.

    Parameters
    ----------
    Q                : int
    P                : int
    m                : int    (default 3)
    layers           : int    1 or 2 (default 1)
    w                : int | None
    theta_electrical : float  Electrical time angle (radians, default 0)

    Returns
    -------
    angles : np.ndarray, shape (Q + 1,)
        Mechanical angles in degrees, 0° … 360° inclusive.
    F      : np.ndarray, shape (Q + 1,)
        Per-unit MMF at each slot boundary (closed staircase).

    Examples
    --------
    >>> angles, F = mmf_waveform(12, 10, layers=2)
    >>> angles.shape, F.shape
    ((13,), (13,))
    >>> abs(F.mean()) < 1e-10
    True
    """
    if w is None:
        w = _optimal_coil_span(Q, P, m, layers)

    n = mmf_distribution(Q, P, m, layers, w, theta_electrical)

    F = np.cumsum(n)
    F -= F.mean()

    angles = np.linspace(0.0, 360.0, Q, endpoint=False)
    angles = np.append(angles, 360.0)
    F = np.append(F, F[0])

    return angles, F

### `mmf_waveform`: Example — Effect of Electrical Time Angle

In [ ]:
from emachines.winding.mmf import mmf_waveform
import numpy as np

print("MMF waveform at three electrical time angles")
print("12-slot / 10-pole double-layer FSCW")
print()

for theta_deg in [0, 30, 90]:
    angles, F = mmf_waveform(12, 10, layers=2,
                              theta_electrical=np.radians(theta_deg))
    print(f"  θ_e = {theta_deg:3d}°  |  peak MMF = {np.max(np.abs(F[:-1])):.4f} pu")

print()
print("Observation: the peak MMF is the same at all time angles (rotating wave).")

---

## `winding_factor_spectrum`: Winding Factor vs. Harmonic Order

### Theory

The winding factor for mechanical harmonic ν is the ratio of the actual phasor sum of conductor voltages to the arithmetic sum (i.e. the ideal case where all conductors are in one slot):

$$
k_{w\nu} = \frac{1}{m} \sum_{k=0}^{m-1}
           \frac{\left|\displaystyle\sum_{i,\,\text{lyr}} s_{i,\text{lyr}}
           \cdot e^{j\,\nu\,\frac{2\pi i}{Q}}\right|}{N_k}
$$

where the inner sum runs over all conductors (across all layers) of phase k, $s_{i,\text{lyr}} \in \{+1,-1\}$ is the conductor direction, and $N_k$ is the total conductor count for phase k.

**Mechanical vs. electrical harmonics:** ν is the *mechanical* harmonic order — the number of complete cycles of the waveform around the full 360° mechanical circumference. The *working harmonic* (the one that produces torque with P poles) is at **ν = p = P/2**.

**Design insight:** Harmonics with high kw(ν) are efficiently produced. For a PMSM, harmonics other than ν = p are parasitic — they store reactive energy and cause rotor losses. A good winding has kw(p) ≈ 0.9–1.0 and kw(ν) ≈ 0 for all other significant ν.

**References:** Bianchi & Bolognani (2002); Müller, Vogt & Ponick (2008), eq. (3.65).

### `winding_factor_spectrum`: Implementation

In [ ]:
# winding_factor_spectrum is defined in emachines.winding.winding_factors
# and imported into this module via the setup cell above.
# It is re-exported from emachines.winding for convenience.
from emachines.winding.winding_factors import winding_factor_spectrum  # noqa: F401

### `winding_factor_spectrum`: Example — Integer-slot vs. FSCW

In [ ]:
from emachines.winding.mmf import winding_factor_spectrum
import numpy as np

configs = [
    dict(Q=24, P=4,  layers=2, label="24s/4p  integer-slot (q=2)"),
    dict(Q=12, P=10, layers=2, label="12s/10p FSCW (q=2/5)"),
    dict(Q=9,  P=8,  layers=2, label="9s/8p   FSCW (q=3/8)"),
]

print(f"{'Harmonic ν':>12}", end="")
for cfg in configs:
    print(f"  {cfg['label']:>28}", end="")
print()
print("-" * 95)

nu_max = 15
nu_arr = None
results = []
for cfg in configs:
    nu_arr, kw = winding_factor_spectrum(cfg["Q"], cfg["P"], layers=cfg["layers"], nu_max=nu_max)
    results.append((cfg, kw))

for i, nu_i in enumerate(nu_arr):
    print(f"  ν = {int(nu_i):>3}     ", end="")
    for cfg_j, kw_j in results:
        val = kw_j[i]
        marker = " ←working" if nu_i == cfg_j["P"] // 2 else ""
        print(f"  {'':>12}kw = {val:.4f}{marker}", end="")
    print()


---

## `mmf_harmonics`: MMF Harmonic Amplitude Spectrum

### Theory

The Fourier decomposition of the airgap MMF gives amplitude coefficients for each spatial harmonic ν. For a winding with winding factor kw(ν) and p pole pairs, the analytical result is:

$$
\hat{F}_\nu = \frac{k_{w\nu} \cdot p}{\nu}
$$

The factor $p/\nu$ arises because the MMF is the spatial integral of the conductor distribution: integrating $\cos(\nu x)$ yields $\sin(\nu x)/\nu$, and multiplying by p normalises the working harmonic (ν = p) to kw₁.

**Three-phase cancellation:** In a balanced m-phase winding, certain harmonics cancel completely. For m = 3, harmonics where ν is a non-working multiple of 3 vanish in the balanced sum. However, the simple rule "ν % m == 0 → cancel" fails when p itself is divisible by m (e.g. 9s/6p where p = 3). This implementation detects cancellation numerically by computing the complex m-phase phasor sum at each ν and checking whether both the forward- and backward-rotating components vanish.

**Mode options:**
- `"three_phase"` — balanced m-phase MMF; zero-sequence harmonics cancelled (default, matches the website tool).
- `"single_phase"` — single-phase MMF; all harmonics retained.
- `"winding_factor"` — returns |kw(ν)| directly without the 1/ν weighting.

**References:** Pyrhönen et al. (2014), §2.4, eq. (2.98); Müller, Vogt & Ponick (2008), eq. (3.65).

### `mmf_harmonics`: Implementation

In [ ]:
#| export
def mmf_harmonics(
    Q: int,
    P: int,
    m: int = 3,
    layers: int = 1,
    w: int | None = None,
    mode: str = "three_phase",
    nu_max: int = 40,
    normalize: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    r"""
    MMF harmonic amplitude spectrum.

    The amplitude of spatial harmonic ν in the airgap MMF:

    .. math::
        \hat{F}_\nu = \frac{k_{w\nu} \cdot p}{\nu}

    For ``mode="three_phase"`` harmonics cancelled by the balanced
    m-phase current sum are identified numerically and zeroed out.

    Parameters
    ----------
    Q      : int
    P      : int
    m      : int    Number of phases (default 3)
    layers : int    (default 1)
    w      : int | None
    mode   : str
        ``"three_phase"``    — balanced m-phase MMF; zero-sequence harmonics
                               cancelled numerically (default).
        ``"single_phase"``   — single-phase MMF; all harmonics retained.
        ``"winding_factor"`` — returns |kw(ν)| with no 1/ν weighting.
    nu_max    : int   Maximum harmonic order (default 40)
    normalize : bool
        ``True``  (default) — divide by the working-harmonic amplitude
                              so the working harmonic = 1.0.
        ``False`` — return raw kw(ν)·p/ν values.

    Returns
    -------
    nu_arr : np.ndarray, shape (nu_max,), dtype int
        Mechanical harmonic orders 1 … nu_max.
    amp    : np.ndarray, shape (nu_max,), dtype float64
        Harmonic amplitudes.

    Examples
    --------
    >>> nu, amp = mmf_harmonics(12, 10, layers=2, normalize=True)
    >>> float(round(amp[4], 4))   # working harmonic (ν=5) should be 1.0
    1.0
    >>> nu, amp = mmf_harmonics(12, 10, layers=2, mode="single_phase",
    ...                         normalize=False)
    >>> amp[4] > 0
    True

    References
    ----------
    Pyrhönen et al. (2014), §2.4, eq. (2.98).
    Müller, Vogt & Ponick (2008), eq. (3.65).
    """
    p = P // 2
    if w is None:
        w = _optimal_coil_span(Q, P, m, layers)

    nu_arr, kw_arr = winding_factor_spectrum(Q, P, m, layers, w, nu_max=nu_max)
    kw_abs = np.abs(kw_arr)

    if mode == "winding_factor":
        amp = kw_abs.copy()
        if normalize:
            fund = kw_abs[p - 1] if p <= nu_max else 0.0
            if fund < 1e-10:
                fund = kw_abs.max() if kw_abs.max() > 1e-10 else 1.0
            amp /= fund
        return nu_arr, amp

    # Raw MMF amplitude: kw(ν)·p / ν
    amp = kw_abs * p / nu_arr.astype(np.float64)

    if mode == "three_phase":
        # Numerically detect harmonics cancelled by the balanced m-phase sum.
        # Forward (positive-sequence) and backward (negative-sequence) phasors
        # are both checked; a harmonic survives if either is non-zero.
        matrix = build_coil_matrix(Q, P, m, layers, w)
        slots = np.arange(Q, dtype=np.float64)

        i_fwd = np.exp(-1j * 2.0 * np.pi * np.arange(m) / m)
        i_bwd = np.exp(+1j * 2.0 * np.pi * np.arange(m) / m)

        for idx, nu in enumerate(nu_arr):
            phasors = np.exp(1j * nu * 2.0 * np.pi * slots / Q)
            T_f = T_b = 0j
            for k in range(m):
                ph = k + 1
                ph_sum = 0j
                for lyr in range(layers):
                    fwd = matrix[lyr] == +ph
                    ret = matrix[lyr] == -ph
                    ph_sum += phasors[fwd].sum() - phasors[ret].sum()
                T_f += i_fwd[k] * ph_sum
                T_b += i_bwd[k] * ph_sum
            if max(abs(T_f), abs(T_b)) < 1e-6:
                amp[idx] = 0.0

    if normalize:
        fund = amp[p - 1] if p <= nu_max else 0.0
        if fund < 1e-10:
            fund = amp.max() if amp.max() > 1e-10 else 1.0
        amp = amp / fund

    return nu_arr, amp

### `mmf_harmonics`: Example — Single-layer vs. Double-layer

In [ ]:
from emachines.winding.mmf import mmf_harmonics
import numpy as np

print("Three-phase MMF spectrum: 12s/10p FSCW")
print("Normalised to working harmonic (ν = 5)")
print()
print(f"{'ν':>4}  {'Single-layer':>14}  {'Double-layer':>14}")
print("-" * 38)

nu1, amp1 = mmf_harmonics(12, 10, layers=1, mode="three_phase", normalize=True)
nu2, amp2 = mmf_harmonics(12, 10, layers=2, mode="three_phase", normalize=True)

for i in range(min(20, len(nu1))):
    if amp1[i] > 1e-4 or amp2[i] > 1e-4:
        marker = " ← working" if nu1[i] == 5 else ""
        print(f"  {int(nu1[i]):>2}  {amp1[i]:>14.4f}  {amp2[i]:>14.4f}{marker}")

print()
print("Observation: double-layer significantly suppresses sub-harmonics.")

### `mmf_harmonics`: Comprehensive comparison — Integer-slot vs. FSCW

In [ ]:
from emachines.winding.mmf import mmf_harmonics, winding_factor_spectrum
import numpy as np

configs = [
    dict(Q=24, P=4,  layers=2, label="24s/4p  integer-slot"),
    dict(Q=12, P=10, layers=2, label="12s/10p FSCW         "),
]

print("Three-phase MMF spectrum comparison (normalised, top harmonics only)")
print()

for cfg in configs:
    nu, amp = mmf_harmonics(cfg["Q"], cfg["P"], layers=cfg["layers"],
                             mode="three_phase", normalize=True)
    p = cfg["P"] // 2
    print(f"  {cfg['label']}  (working harmonic ν = {p})")
    significant = [(int(nu[i]), amp[i]) for i in range(len(nu)) if amp[i] > 0.01]
    significant.sort(key=lambda x: -x[1])
    for h, a in significant[:8]:
        marker = " ← working" if h == p else ""
        print(f"    ν = {h:>3}:  {a:.4f}{marker}")
    print()

---

## References

1. Pyrhönen, J., Jokinen, T., & Hrabovcová, V. (2014). *Design of Rotating Electrical Machines*, 2nd ed. Wiley. §2.4, eq. (2.73), (2.98).
2. Bianchi, N., & Bolognani, S. (2002). Design techniques for reducing cogging torque in surface-mounted PM motors. *IEEE Transactions on Industry Applications*, 38(5), 1259–1265. DOI: [10.1109/TIA.2002.802909](https://doi.org/10.1109/TIA.2002.802909)
3. Müller, G., Vogt, K., & Ponick, B. (2008). *Berechnung elektrischer Maschinen*. Wiley-VCH. §3.4–3.6, eq. (3.65).
4. SWAT-EM open-source reference implementation: [https://github.com/bayonet222/swat-em](https://github.com/bayonet222/swat-em) — accessed 2026-06-22.

---

## Tests

In [ ]:
#| hide
import math
import numpy as np
from emachines.winding.mmf import (
    mmf_distribution,
    mmf_waveform,
    winding_factor_spectrum,
    mmf_harmonics,
)

# ── mmf_distribution ─────────────────────────────────────────────────────────

# Shape
n = mmf_distribution(12, 10, layers=2)
assert n.shape == (12,), f"Expected shape (12,), got {n.shape}"
print("✓ mmf_distribution: shape")

# Zero net MMF around airgap (Ampere's law: closed path sum = 0)
assert abs(n.sum()) < 1e-10, f"Net MMF = {n.sum():.2e}, expected ≈ 0"
print("✓ mmf_distribution: zero net MMF (Ampere's law)")

# Time symmetry: θ_e and θ_e + π should give equal-and-opposite distributions
n0  = mmf_distribution(12, 10, layers=2, theta_electrical=0.0)
npi = mmf_distribution(12, 10, layers=2, theta_electrical=math.pi)
assert np.allclose(n0, -npi, atol=1e-12), "Half-period symmetry failed"
print("✓ mmf_distribution: half-period sign symmetry")

# Single-layer: check shape and zero-sum for a standard integer-slot winding
n_sl = mmf_distribution(24, 4, layers=1)
assert n_sl.shape == (24,), f"Expected (24,), got {n_sl.shape}"
assert abs(n_sl.sum()) < 1e-10, "Single-layer 24s/4p net MMF ≠ 0"
print("✓ mmf_distribution: 24s/4p single-layer")


# ── mmf_waveform ──────────────────────────────────────────────────────────────

# Shape: (Q+1,) for closed staircase
angles, F = mmf_waveform(12, 10, layers=2)
assert angles.shape == (13,), f"Expected (13,), got {angles.shape}"
assert F.shape == (13,), f"Expected (13,), got {F.shape}"
print("✓ mmf_waveform: shape")

# First and last values equal (closed staircase)
assert math.isclose(F[0], F[-1], abs_tol=1e-12), "Staircase not closed"
print("✓ mmf_waveform: closed staircase")

# Zero mean (DC offset removed)
# The mean of the closed staircase includes the repeated endpoint; check
# mean of the open part (first Q points)
assert abs(F[:-1].mean()) < 1e-10, f"Non-zero mean: {F[:-1].mean():.2e}"
print("✓ mmf_waveform: zero mean (DC removed)")

# Angle range: 0° to 360°
assert math.isclose(angles[0], 0.0) and math.isclose(angles[-1], 360.0), \
    f"Angle range wrong: [{angles[0]}, {angles[-1]}]"
print("✓ mmf_waveform: angle range 0°–360°")

# Integer-slot winding
angles2, F2 = mmf_waveform(24, 4, layers=2)
assert angles2.shape == (25,) and F2.shape == (25,)
assert abs(F2[:-1].mean()) < 1e-10
print("✓ mmf_waveform: 24s/4p double-layer")


# ── winding_factor_spectrum ───────────────────────────────────────────────────

# Shape
nu, kw = winding_factor_spectrum(12, 10, layers=2, nu_max=20)
assert nu.shape == (20,) and kw.shape == (20,), "Shape mismatch"
print("✓ winding_factor_spectrum: shape")

# All kw in [0, 1]
assert np.all(kw >= -1e-10) and np.all(kw <= 1.0 + 1e-10), \
    f"kw out of [0,1]: min={kw.min():.4f}, max={kw.max():.4f}"
print("✓ winding_factor_spectrum: kw ∈ [0, 1]")

# 12s/10p working harmonic ν=5: kw ≈ 0.933 (known reference value)
kw_working = float(kw[4])   # ν=5 is at index 4 (1-indexed)
assert math.isclose(kw_working, 0.9330, abs_tol=1e-3), \
    f"12s/10p kw(ν=5) = {kw_working:.4f}, expected ≈ 0.9330"
print(f"✓ winding_factor_spectrum: 12s/10p kw(ν=5) = {kw_working:.4f} ≈ 0.9330")

# 12s/8p working harmonic ν=4: kw ≈ 0.866 (known reference value)
_, kw8 = winding_factor_spectrum(12, 8, layers=2, nu_max=10)
assert math.isclose(float(kw8[3]), 0.8660, abs_tol=1e-3), \
    f"12s/8p kw(ν=4) = {kw8[3]:.4f}, expected ≈ 0.8660"
print(f"✓ winding_factor_spectrum: 12s/8p  kw(ν=4) = {float(kw8[3]):.4f} ≈ 0.8660")

# 12s/4p full-pitch (q=1): kw(ν=2) should be 1.0
_, kw12 = winding_factor_spectrum(12, 4, layers=1, w=6, nu_max=10)
assert math.isclose(float(kw12[1]), 1.0, abs_tol=1e-4), \
    f"24s/4p full-pitch kw(ν=2) = {kw12[1]:.4f}, expected 1.0"
print(f"✓ winding_factor_spectrum: 12s/4p  kw(ν=2) = {float(kw12[1]):.4f} (q=1, full-pitch)")


# ── mmf_harmonics ─────────────────────────────────────────────────────────────

# Shape
nu, amp = mmf_harmonics(12, 10, layers=2, nu_max=20)
assert nu.shape == (20,) and amp.shape == (20,), "Shape mismatch"
print("✓ mmf_harmonics: shape")

# Normalised: working harmonic = 1.0
nu, amp_norm = mmf_harmonics(12, 10, layers=2, normalize=True)
assert math.isclose(float(amp_norm[4]), 1.0, abs_tol=1e-6), \
    f"Normalised working harmonic = {amp_norm[4]:.6f}, expected 1.0"
print("✓ mmf_harmonics: working harmonic normalised to 1.0")

# Non-negative amplitudes
assert np.all(amp_norm >= -1e-10), f"Negative amplitude: {amp_norm.min():.4f}"
print("✓ mmf_harmonics: non-negative amplitudes")

# Three-phase mode: ν=1 sub-harmonic for 12s/10p DL should be present
# (it is not cancelled by 3-phase sum since it's the sub-harmonic)
nu_3ph, amp_3ph = mmf_harmonics(12, 10, layers=2, mode="three_phase", normalize=False)
assert amp_3ph[0] > 1e-6, "Sub-harmonic (ν=1) should survive in 12s/10p DL"
print("✓ mmf_harmonics: ν=1 sub-harmonic present in 12s/10p DL three_phase")

# Single-phase mode: more harmonics than three-phase
_, amp_sp = mmf_harmonics(12, 10, layers=2, mode="single_phase", normalize=False)
nnz_3ph = int(np.sum(amp_3ph > 1e-6))
nnz_sp  = int(np.sum(amp_sp > 1e-6))
assert nnz_sp >= nnz_3ph, \
    f"single_phase ({nnz_sp} harmonics) should have ≥ three_phase ({nnz_3ph})"
print(f"✓ mmf_harmonics: single_phase ({nnz_sp}) ≥ three_phase ({nnz_3ph}) non-zero harmonics")

# winding_factor mode: working harmonic matches winding_factor_spectrum
_, amp_wf = mmf_harmonics(12, 10, layers=2, mode="winding_factor", normalize=False)
_, kw_ref = winding_factor_spectrum(12, 10, layers=2, nu_max=40)
assert np.allclose(amp_wf, np.abs(kw_ref), atol=1e-10), \
    "winding_factor mode should match winding_factor_spectrum"
print("✓ mmf_harmonics: winding_factor mode matches winding_factor_spectrum")

# Integer-slot 24s/4p: half-wave symmetry cancels even electrical harmonics.
# Present: ν_mech = p*(odd_elec) = 2,6,10,... Cancelled: p*(even_elec) = 4,8,12,...
_, amp_int = mmf_harmonics(24, 4, layers=2, mode="three_phase", normalize=False, nu_max=15)
assert math.isclose(amp_int[3], 0.0, abs_tol=1e-6), \
    f"ν=4 (2nd elec, even) must cancel in 24s/4p: {amp_int[3]:.2e}"
assert math.isclose(amp_int[7], 0.0, abs_tol=1e-6), \
    f"ν=8 (4th elec, even) must cancel in 24s/4p: {amp_int[7]:.2e}"
assert amp_int[5] > 0.01, \
    f"ν=6 (3rd elec, odd) must be present in 24s/4p: {amp_int[5]:.4f}"
print("✓ mmf_harmonics: even-elec harmonics (ν=4,8) cancelled, odd (ν=6) present in 24s/4p")

print()
print("✓ All winding MMF tests passed")

# ── Parameterised MMF checks ──────────────────────────────────────────────────
# For each configuration: shape/symmetry invariants + physics-based harmonic assertions.

# 12s/10p FSCW (p=5): odd harmonics only; sub-harmonic nu=1 present (FSCW trait);
#   backward harmonic nu=7 present; all even harmonics cancelled.
Q, P, layers = 12, 10, 2
n = mmf_distribution(Q, P, layers=layers)
assert n.shape == (Q,) and abs(n.sum()) < 1e-10
_, F = mmf_waveform(Q, P, layers=layers)
assert abs(F[:-1].mean()) < 1e-10
_, amp = mmf_harmonics(Q, P, layers=layers, mode="three_phase", normalize=True)
assert math.isclose(amp[4], 1.0,    abs_tol=1e-4),  "12s/10p: working nu=5 not 1.0"
assert amp[0]  > 0.30,               "12s/10p: sub-harmonic nu=1 must be present (FSCW)"
assert amp[6]  > 0.60,               "12s/10p: backward nu=7 must be present"
assert math.isclose(amp[3], 0.0, abs_tol=1e-6),  "12s/10p: nu=4 must be cancelled"
assert math.isclose(amp[5], 0.0, abs_tol=1e-6),  "12s/10p: nu=6 must be cancelled"
print("pass  12s/10p: sub(nu=1)={:.3f}  work(nu=5)={:.3f}  back(nu=7)={:.3f}  even=0".format(
    amp[0], amp[4], amp[6]))

# 12s/14p FSCW (p=7): even harmonics cancelled; parasitic nu=5 EXCEEDS working
#   harmonic — key quality-flag for this over-poled winding.
Q, P, layers = 12, 14, 2
n = mmf_distribution(Q, P, layers=layers)
assert abs(n.sum()) < 1e-10
_, amp = mmf_harmonics(Q, P, layers=layers, mode="three_phase", normalize=True)
assert math.isclose(amp[6], 1.0,    abs_tol=1e-4), "12s/14p: working nu=7 not 1.0"
assert amp[4] > 1.3,                 "12s/14p: parasitic nu=5 should exceed working (amp>1.3)"
assert amp[0] > 0.40,                "12s/14p: sub-harmonic nu=1 present (FSCW)"
assert math.isclose(amp[5], 0.0, abs_tol=1e-6), "12s/14p: nu=6 must be cancelled"
assert math.isclose(amp[7], 0.0, abs_tol=1e-6), "12s/14p: nu=8 must be cancelled"
print("pass  12s/14p: parasitic(nu=5)={:.3f} > working(nu=7)=1.0  even=0".format(amp[4]))

# 12s/8p FSCW (p=4): cleanest FSCW — only multiples of 4 survive; nu=12 (3*p,
#   triplen electrical) cancelled; harmonic ratio nu=8/nu=4 = 0.5.
Q, P, layers = 12, 8, 2
n = mmf_distribution(Q, P, layers=layers)
assert abs(n.sum()) < 1e-10
_, amp = mmf_harmonics(Q, P, layers=layers, mode="three_phase", normalize=True, nu_max=20)
assert math.isclose(amp[3], 1.0,    abs_tol=1e-4), "12s/8p: working nu=4 not 1.0"
assert math.isclose(amp[7], 0.5,    abs_tol=1e-3), "12s/8p: nu=8 should be 0.5 of working"
assert math.isclose(amp[11], 0.0,   abs_tol=1e-6), "12s/8p: nu=12 (triplen elec) must cancel"
for nu_check in [1, 2, 3, 5, 6, 7, 9, 10]:
    assert math.isclose(amp[nu_check-1], 0.0, abs_tol=1e-6),         f"12s/8p: nu={nu_check} must be cancelled, got {amp[nu_check-1]:.4f}"
print("pass  12s/8p: work(nu=4)=1.0  nu=8={:.3f}  nu=12=0  non-multiples-of-4=0".format(amp[7]))

# 24s/4p ISW (p=2, q=2): integer-slot; sub-harmonics absent; even electrical
#   harmonics absent; nu=6 (3rd electrical, odd) is present.
Q, P, layers = 24, 4, 2
n = mmf_distribution(Q, P, layers=layers)
assert abs(n.sum()) < 1e-10
_, amp = mmf_harmonics(Q, P, layers=layers, mode="three_phase", normalize=True, nu_max=12)
assert math.isclose(amp[1], 1.0,    abs_tol=1e-4), "24s/4p: working nu=2 not 1.0"
assert math.isclose(amp[0], 0.0,    abs_tol=1e-6), "24s/4p: sub-harmonic nu=1 must be absent (ISW)"
assert math.isclose(amp[2], 0.0,    abs_tol=1e-6), "24s/4p: nu=3 cancelled"
assert math.isclose(amp[3], 0.0,    abs_tol=1e-6), "24s/4p: nu=4 (even elec) cancelled"
assert amp[5] > 0.20,                "24s/4p: nu=6 (3rd elec harmonic) should be present"
print("pass  24s/4p: work(nu=2)=1.0  nu=1,3,4=0  nu=6={:.3f}".format(amp[5]))

# 36s/8p ISW (p=4, q=1.5): fractional-slot ISW; sub-harmonics absent;
#   chording (w=4 vs tau_p=4.5) strongly suppresses nu=8 (2nd elec harmonic).
Q, P, layers = 36, 8, 2
n = mmf_distribution(Q, P, layers=layers)
assert abs(n.sum()) < 1e-10
_, amp = mmf_harmonics(Q, P, layers=layers, mode="three_phase", normalize=True, nu_max=20)
assert math.isclose(amp[3], 1.0,    abs_tol=1e-4), "36s/8p: working nu=4 not 1.0"
assert math.isclose(amp[0], 0.0,    abs_tol=1e-6), "36s/8p: nu=1 must be absent (ISW)"
for nu_check in [1, 2, 3, 5, 6, 7, 9, 10, 11]:
    assert math.isclose(amp[nu_check-1], 0.0, abs_tol=1e-6),         f"36s/8p: nu={nu_check} must cancel, got {amp[nu_check-1]:.4f}"
assert amp[7] < 0.05,                "36s/8p: nu=8 should be strongly suppressed by chording"
assert amp[11] > 0.15,               "36s/8p: nu=12 (3rd elec) should be present"
print("pass  36s/8p: work(nu=4)=1.0  nu=8={:.4f} (chording suppressed)  nu=12={:.3f}".format(
    amp[7], amp[11]))
